In [1]:
from oqd_compiler_infrastructure import Post, PrettyPrint
from oqd_core.analysis.analog.cfg import AnalogCFGBuilder
from oqd_core.analysis.analog.type_checker import AnalogTypeChecker
from oqd_core.frontend.analog import parse_analog
from oqd_core.analysis.analog.symbol_table import AnalogSymbolTableBuilder
from oqd_core.compiler.analog.passes.compile import compile_analog_circuit
from oqd_analog_emulator.interpreter import QutipInterpreter

printer = Post(PrettyPrint())

source = """
a=2 \n b = a + 3 \n H = %I %* %X \n c = true \n d = not c \n 
e = c and d \n f = c or d \n g = a <= b \n h = a >= b \n 
i = a == b \n k = a != b \n l = a - 1 \n m = a * b \n n = 2^3 \n
"""

source = """ 
a = 2 \n b = 5 \n 
if (a > 0) { \n b = 3 } \n
if (b < 0) { \n a = 5} \n
else { \n a = 10}
if (a < 0) { \n b = 10}
"""

source = """ 
n = 5 \n
while (n > 0) { \n n = n - 1}
"""

source = """
a = [2, 3, 4]
"""

source = """ 
r = qreg(2)
q0 = r[0]
q1 = r[1]
"""


source = """ 
a = sin(3.14 / 4)
b = abs(-5)
c = atan2(8, 5)
d = heaviside(2)
i = -4 + 4 * 1j
e = real(0 + 1j)
f = imag(1j)
g = conj(1 + 1j)
"""

source = """ 
r = qreg(2)
q0 = r[0]
q1 = r[1]
list = [1, 2, 3, 4]
b = list
initialize(q0)
// initialize(q1)
result = evolve(-(3.14 * #t / 4) %* %X , 1, r[0])
// result2 = evolve(-(3.14 / 4) %* %X , 1, q1)
result = evolve(-(3.14 / 4) %* (%X %* %X) , 1, r)
measurement = measure(q0)
"""

# source = """ 
# r = qreg(2)
# initialize(r)
# hamiltonian = -(3.14 * #t / 4) %* %X
# result = evolve(hamiltonian, 1, r[0])
# h2 = #s %* %X
# result2 = evolve(h2, 1, r[1])
# """

source = """ 
r = qreg(5)
q0 = r[0]
q1 = r[1]
q3 = r[3]
initialize(r)
s = [q0, q1]
// s = [q1, q0]
// t = [q1, q3]
result = evolve(%X , 1, q0)
result2 = evolve( %X %@ %X, 1, s)
// result2 = evolve( %X %@ %X, 1, t)
// evolve(%X, 1, r[2])

"""
# source = """ 

# a = 1
# b = 2

# [q0,a]
# 1+2
# """

# source = """ 
# r = qreg(5)
# 1+2
# q0 = r[0]
# q1 = r[1]
# s = [q0, q1]
# """



# source = "a = (#t + 4) * 2"

circuit = parse_analog(source)
cfg = AnalogCFGBuilder().run(circuit)
checker = AnalogTypeChecker(cfg)

symbol_analysis = AnalogSymbolTableBuilder(cfg, checker.dataflow_result)
symbol_table = symbol_analysis.symbol_table

circuit, cfg = compile_analog_circuit(circuit=circuit, cfg=cfg, symbol_table=symbol_table)


In [2]:
interpreter = QutipInterpreter(graph=cfg)
interpreter.run()

[<ListTerminators.LISTSTART: 0>, <ListTerminators.LISTEND: 1>]

In [3]:
instructions = interpreter.get_instructions()
instructions

[[('QREG', 'r', 5)],
 [('GLOBAL', 'q0'), ('EXTRACT', 'r', 0), ('STORE', 'q0')],
 [('GLOBAL', 'q1'), ('EXTRACT', 'r', 1), ('STORE', 'q1')],
 [('GLOBAL', 'q3'), ('EXTRACT', 'r', 3), ('STORE', 'q3')],
 [('LOAD', 'r'), ('INIT',)],
 [('GLOBAL', 's'),
  ('CONST', <ListTerminators.LISTEND: 1>),
  ('LOAD', 'q1'),
  ('LOAD', 'q0'),
  ('CONST', <ListTerminators.LISTSTART: 0>),
  ('STORE', 's')],
 [('GLOBAL', 'result'),
  ('LOAD', 'q0'),
  ('CONST', 1),
  ('CONST',
   Quantum object: dims=[[2], [2]], shape=(2, 2), type='oper', dtype=CSR, isherm=True
   Qobj data =
   [[0. 1.]
    [1. 0.]]),
  ('EVOLVE',),
  ('STORE', 'result')],
 [('GLOBAL', 'result2'),
  ('LOAD', 's'),
  ('CONST', 1),
  ('CONST',
   Quantum object: dims=[[2], [2]], shape=(2, 2), type='oper', dtype=CSR, isherm=True
   Qobj data =
   [[0. 1.]
    [1. 0.]]),
  ('CONST',
   Quantum object: dims=[[2], [2]], shape=(2, 2), type='oper', dtype=CSR, isherm=True
   Qobj data =
   [[0. 1.]
    [1. 0.]]),
  ('KRON',),
  ('EVOLVE',),
  ('STOR

In [4]:
store = interpreter.status()
store
# interpreter.VM.stack
# store['q0'].state

{'r': [<ListTerminators.LISTSTART: 0>,
  RegisterObject(name='r', index=0),
  RegisterObject(name='r', index=1),
  RegisterObject(name='r', index=2),
  RegisterObject(name='r', index=3),
  RegisterObject(name='r', index=4),
  <ListTerminators.LISTEND: 1>],
 'q0': RegisterObject(name='r', index=0),
 'q1': RegisterObject(name='r', index=1),
 'q3': RegisterObject(name='r', index=3),
 's': [<ListTerminators.LISTSTART: 0>,
  RegisterObject(name='r', index=0),
  RegisterObject(name='r', index=1),
  <ListTerminators.LISTEND: 1>],
 'result': [<ListTerminators.LISTSTART: 0>, <ListTerminators.LISTEND: 1>],
 'result2': [<ListTerminators.LISTSTART: 0>, <ListTerminators.LISTEND: 1>]}

In [5]:
# from oqd_core.compiler.analog.math.rules import SubstituteMathVar
# from oqd_core.interface.analog.expr import MathVar
# from oqd_compiler_infrastructure import Post
# substitute_pass = Post(SubstituteMathVar(MathVar(class_='MathVar', name='#s'), MathVar(class_='MathVar', name='#t') - 10))

# substitute_pass(store['a'])
registers = interpreter.VM.registers
registers

{RegisterObject(name='r', index=0): <oqd_analog_emulator.interpreter.QubitRegister at 0x10bdf7a50>,
 RegisterObject(name='r', index=1): <oqd_analog_emulator.interpreter.QubitRegister at 0x10bdf7a50>,
 RegisterObject(name='r', index=2): <oqd_analog_emulator.interpreter.QubitObject at 0x11a8592d0>,
 RegisterObject(name='r', index=3): <oqd_analog_emulator.interpreter.QubitObject at 0x11a859410>,
 RegisterObject(name='r', index=4): <oqd_analog_emulator.interpreter.QubitObject at 0x11a859c50>}

In [6]:
cfg.to_dict()

{0: {'register_id': 0,
  'kind': 'start',
  'stmt': {},
  'preds': [],
  'succs': [1],
  'exit_nodes': [],
  'edge_labels': {}},
 1: {'register_id': 1,
  'kind': 'stmt',
  'stmt': {'class_': 'Declaration',
   'name': 'r',
   'value': {'class_': 'QuantumRegister', 'size': 5}},
  'preds': [0],
  'succs': [2],
  'exit_nodes': [],
  'edge_labels': {}},
 2: {'register_id': 2,
  'kind': 'stmt',
  'stmt': {'class_': 'Declaration',
   'name': 'q0',
   'value': {'class_': 'Extract',
    'access': {'class_': 'Access', 'name': 'r'},
    'index': 0}},
  'preds': [1],
  'succs': [3],
  'exit_nodes': [],
  'edge_labels': {}},
 3: {'register_id': 3,
  'kind': 'stmt',
  'stmt': {'class_': 'Declaration',
   'name': 'q1',
   'value': {'class_': 'Extract',
    'access': {'class_': 'Access', 'name': 'r'},
    'index': 1}},
  'preds': [2],
  'succs': [4],
  'exit_nodes': [],
  'edge_labels': {}},
 4: {'register_id': 4,
  'kind': 'stmt',
  'stmt': {'class_': 'Declaration',
   'name': 'q3',
   'value': {'cla